本ノートブックは arman-bd 氏が公開している実在の極小 LLM **GuppyLM** (MIT ライセンス、<https://github.com/arman-bd/guppylm>) を参考に、そのモデル定義・学習・推論を忠実に再実装したものである
ライセンスおよびオリジナル著者のクレジットは末尾の参考文献に記載する

# GuppyLM ― 実在する 8.7M パラメータの極小チャットモデルを再実装する

本章では、GPT 系 Transformer 言語モデルの内部構造を「自分の手で動かして理解する」ために、実在する超小型チャットモデル **GuppyLM** を PyTorch でゼロから再実装する
- GuppyLM は arman-bd 氏が 2025 年に公開した教育目的のデコーダのみ Transformer であり、パラメータ数はわずか **8.7M**、語彙は 4096、文脈長は 128 と極端に小さい
- それでも ChatML 形式の対話データで SFT することで、グッピー(熱帯魚)になりきった短い会話が成立する

本ノートブックではこのモデルを Colab 上で数分〜十数分程度で学習し、ipywidgets によるチャット UI でその振る舞いを体験する

## 全体の流れ

- 背景 ― GuppyLM とは何か、なぜ 8.7M なのか
- 環境準備 ― 必要なライブラリのインストール
- トークナイザ ― 公式リポジトリの `tokenizer.json` をダウンロードして読み込む
- データセット ― Hugging Face Hub から `arman-bd/guppylm-60k-generic` を取得する
- ChatML 整形 ― `<|im_start|>` 形式のプロンプトに変換する
- `Dataset` / `DataLoader` ― 固定長 128 の causal LM 用バッチを作る
- モデル定義 ― pre-norm Transformer、ReLU FFN、weight tying
- 設定とインスタンス化 ― 8.7M 構成を組み立ててパラメータ数を確認する
- 学習ループ ― AdamW + cosine + AMP で最大 10000 ステップ
- 評価 ― loss と perplexity を確認する
- `chat_completion` ― 公式 `inference.py` の関数を再現する
- サンプル対話 と ipywidgets UI / CUI フォールバック
- **LLM 効率化手法** ― Attention 効率化、KV キャッシュ削減、システム最適化、モデル圧縮、MoE、LoRA 系 PEFT、コンテキスト長拡張、ハイブリッドアーキテクチャ
- まとめと参考文献

## 背景 ― GuppyLM とは

**GuppyLM** は arman-bd 氏が公開している教育目的の極小 LLM である

名前の通り「グッピー(熱帯魚)」を人格として与えられており、自分をグッピーだと思い込んで短い会話を返す

公式リポジトリの README によれば、設計上のポイントは次の通りである

- **単一バリアント**しか存在しない。パラメータ数は約 **8.7M**(`d_model=384`、`n_layers=6`、`n_heads=6`、`ffn_hidden=768`、`vocab_size=4096`、`max_seq_len=128`)。大・中・小のような系列は無く、この 1 構成のみ
- アーキテクチャは GPT 系のデコーダのみ Transformer で、**pre-norm LayerNorm**、**ReLU 活性**の 2 層 FFN、学習可能な絶対位置埋め込み、入出力の **weight tying**、`dropout=0.1` を利用
- トークナイザは BPE (`tokenizers` ライブラリ) で語彙 4096、特殊トークンは `<|pad|>` (0)、`<|im_start|>` (1)、`<|im_end|>` (2) に対応する ID を割り当て
- 学習データは `arman-bd/guppylm-60k-generic` という 6 万件規模の QA ペアで、ChatML 形式に整形して因果言語モデリング損失で学習
- 損失計算では `pad` を `ignore_index` として除外
- ライセンスは **MIT**
本ノートブックのコードも同ライセンスの範囲でオリジナル実装を参考にしている

8.7M というサイズは、昨今の Llama 4・Qwen 3・Gemma 3 といった数十〜数百 B クラスの最前線 LLM と比較すると 4〜5 桁小さい

しかし「Transformer がどのように token を予測するか」「SFT でどのように口調が変わるか」を自分のマシンで完結して観察するには十分であり、教育用途には理想的な規模と言える

## 環境準備

Colab 環境を想定し、必要なライブラリをインストールする
`torch` は Colab に既に入っているので、ここでは `tokenizers`、`datasets`、`ipywidgets` のみを入れる

In [1]:
!pip install -q tokenizers datasets ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.4 MB/s eta 0:00:00


In [2]:
import os, math, time, json, random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device =', device)
print('torch  =', torch.__version__)

SEED = 1234
random.seed(SEED); torch.manual_seed(SEED)
if device == 'cuda':
    torch.cuda.manual_seed_all(SEED)

device = cuda
torch  = 2.10.0+cu128


## トークナイザのダウンロード

GuppyLM 公式リポジトリの `docs/tokenizer.json` を直接 `wget` で取得し、`tokenizers.Tokenizer.from_file` で読み込む。これにより公式モデルと完全に同一の BPE マージ規則・特殊トークン ID を使える

In [3]:
!wget -q -O tokenizer.json https://raw.githubusercontent.com/arman-bd/guppylm/main/docs/tokenizer.json

from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file('tokenizer.json')

VOCAB_SIZE = tokenizer.get_vocab_size()
print('vocab size =', VOCAB_SIZE)

# 特殊トークン ID を取得する。GuppyLM 既定は pad=0, im_start=1, im_end=2。
PAD_ID      = tokenizer.token_to_id('<|pad|>') or 0
BOS_ID      = tokenizer.token_to_id('<|im_start|>')
EOS_ID      = tokenizer.token_to_id('<|im_end|>')
print('PAD, BOS, EOS =', PAD_ID, BOS_ID, EOS_ID)

sample = 'Hello, I am a guppy.'
enc = tokenizer.encode(sample)
print('ids    =', enc.ids)
print('tokens =', enc.tokens)

vocab size = 2418
PAD, BOS, EOS = 0 1 2
ids    = [632, 65, 374, 83, 278, 7]
tokens = ['ello', 'Ġ', 'Ġam', 'Ġa', 'Ġguppy', '.']


## データセットの取得

`arman-bd/guppylm-60k-generic` は、`instruction` / `input` / `output` 形式に近い汎用 QA ペア約 6 万件からなる小規模データセットである

`datasets` ライブラリで Hub から直接ロードする

In [4]:
from datasets import load_dataset

ds = load_dataset('arman-bd/guppylm-60k-generic')
print(ds)
train_raw = ds['train']
print('num examples =', len(train_raw))
print('first example:')
print(train_raw[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/557k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/56.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/57000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'category'],
        num_rows: 57000
    })
    test: Dataset({
        features: ['input', 'output', 'category'],
        num_rows: 3000
    })
})
num examples = 57000
first example:
{'input': 'what about a tankmate', 'output': "would it live along the glass wall. that's my spot.", 'category': 'snail'}


## ChatML 整形

GuppyLM は ChatML 形式のプロンプトで学習される。1 件のサンプルは次の文字列にまとめられる

```
<|im_start|>user\n{user_message}<|im_end|>\n<|im_start|>assistant\n{assistant_message}<|im_end|>
```

`instruction` と `input` が両方存在する場合は、改行で連結して user 側メッセージとする

In [5]:
def build_user_message(example):
    instr = (example.get('instruction') or '').strip()
    inp   = (example.get('input') or '').strip()
    if inp:
        return f'{instr}\n{inp}' if instr else inp
    return instr

def format_chatml(example):
    user = build_user_message(example)
    assistant = (example.get('output') or '').strip()
    return (
        f'<|im_start|>user\n{user}<|im_end|>\n'
        f'<|im_start|>assistant\n{assistant}<|im_end|>'
    )

print(format_chatml(train_raw[0])[:400])

<|im_start|>user
what about a tankmate<|im_end|>
<|im_start|>assistant
would it live along the glass wall. that's my spot.<|im_end|>


## `Dataset` と `DataLoader`

最大長 `max_seq_len=128` で切り詰め、短い系列は `<|pad|>` でパディングする

因果 LM なので入力と正解は 1 トークンずらしたものを使う。損失計算では `ignore_index=PAD_ID` を指定するため、パディング部分はそのまま 0 で埋めて構わない

In [6]:
MAX_SEQ_LEN = 128

class GuppyChatDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_seq_len=128):
        self.data = hf_split
        self.tok = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = format_chatml(self.data[idx])
        ids = self.tok.encode(text).ids[: self.max_seq_len]
        if len(ids) < self.max_seq_len:
            ids = ids + [PAD_ID] * (self.max_seq_len - len(ids))
        ids = torch.tensor(ids, dtype=torch.long)
        x = ids[:-1]
        y = ids[1:]
        return x, y

train_ds = GuppyChatDataset(train_raw, tokenizer, MAX_SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0, drop_last=True)

xb, yb = next(iter(train_loader))
print('xb', xb.shape, 'yb', yb.shape)

xb torch.Size([32, 127]) yb torch.Size([32, 127])


## モデル定義

ここからがGuppyLM 公式 `guppylm/model.py` の実装をほぼそのまま Python/PyTorch で書き下した本体である

構成は次の通り
- `TokenEmbedding` + 学習可能な `PositionalEmbedding`
- `n_layers` 個の `TransformerBlock`
- 各ブロックは **pre-norm**、つまり `x + MHA(LN(x))` と `x + FFN(LN(x))` の形
- `MultiHeadSelfAttention` は因果マスク付き
- `FeedForward` は 2 層 MLP で活性は **ReLU**、中間次元は `ffn_hidden=768`
- 最終 `LayerNorm` の後に語彙射影。重みは入力埋め込みと **tying** する

In [7]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=True)
        self.proj = nn.Linear(d_model, d_model, bias=True)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        att = att.masked_fill(mask, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_drop(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_drop(self.proj(y))
        return y

class FeedForward(nn.Module):
    def __init__(self, d_model, ffn_hidden, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, ffn_hidden)
        self.fc2 = nn.Linear(ffn_hidden, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(self.fc2(F.relu(self.fc1(x))))

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_hidden, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, ffn_hidden, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class GuppyLM(nn.Module):
    def __init__(self, vocab_size, d_model=384, n_layers=6, n_heads=6,
                 ffn_hidden=768, max_seq_len=128, dropout=0.1, pad_id=0):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.pad_id = pad_id
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ffn_hidden, dropout)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        # weight tying
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.max_seq_len
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=self.pad_id,
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, eos_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(1e-8, temperature)
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if eos_id is not None and next_id.item() == eos_id:
                break
        return idx

## コラム: なぜ GuppyLM は SwiGLU でなく ReLU なのか

近年の主要 LLM (Llama 系、Qwen 系、Gemma 系、Mistral など) は FFN に **SwiGLU** を採用しているのに対し、GuppyLM は素朴な **ReLU 二層 FFN** を採用している

### 理由
1. **教育目的の最小実装**
- GuppyLM の README にも "vanilla transformer" と明記されており、GQA、RoPE、SwiGLU、早期終了などの近代的最適化を**意図的にすべて省いている**
- 学習者が Vaswani et al. (2017) の原論文と1対1で対応づけられる構造を優先している
2. **8.7M という規模では効果が小さい**
- SwiGLU の優位性が明確に出るのは数億〜数十億パラメータ以上であり、`ffn_hidden=768` の小型 FFN ではゲインは誤差レベルにとどまる
3. **パラメータ予算**
- SwiGLU は gate / up / down の 3 線形層を持つため、同じ `ffn_hidden` だと素の ReLU FFN (2 層) より約 1.5 倍パラメータが増える
- 総パラメータ 8.7M を維持するには `ffn_hidden` を 768 → 約 512 に削る必要があり、そこまでして得る効果は小さい

### SwiGLU に変えると良くなるか
**生成品質**:
- 60K サンプル (実質約 200 万トークン) という小データセットでは、ほぼ差は出ない可能性が高い
- Shazeer (2020) "GLU Variants Improve Transformer" でも、改善幅はモデル規模と学習トークン数に強く依存し、~10M 規模・10K ステップでは perplexity 改善は数 % 程度にとどまる例が多い
**学習安定性**:
- SwiGLU は LayerNorm + 残差と相性が良く、収束がやや滑らかになる傾向はある
**コスト**:
- 同じ `ffn_hidden=768` のままなら 1 ブロックあたり約 30 万パラメータ増、6 ブロックで約 180 万増 → 8.7M → 約 10.5M
Colab T4 では学習時間は数十秒〜1 分しか変わらない

教育用ミニチャットモデルという GuppyLM の目的では、ReLU で十分

**「同じレシピでアーキテクチャ近代化の効果を体感する」教材として** SwiGLU 版を比較実装するのは興味深い拡張

参考までに、SwiGLU の典型的な実装は以下のようになる

```python
class SwiGLUFFN(nn.Module):
    def __init__(self, d_model, ffn_hidden, dropout=0.1):
        super().__init__()
        self.w_gate = nn.Linear(d_model, ffn_hidden, bias=False)
        self.w_up   = nn.Linear(d_model, ffn_hidden, bias=False)
        self.w_down = nn.Linear(ffn_hidden, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))
```

`Block` の中で `FFN` をこの `SwiGLUFFN` に差し替えるだけで近代的な FFN に移行できる

## 設定とインスタンス化

公式の 8.7M 構成をそのまま使ってモデルを作り、パラメータ数を確認する。weight tying のため `head.weight` は重複カウントしない

In [8]:
config = dict(
    vocab_size = VOCAB_SIZE,
    d_model    = 384,
    n_layers   = 6,
    n_heads    = 6,
    ffn_hidden = 768,
    max_seq_len= MAX_SEQ_LEN,
    dropout    = 0.1,
    pad_id     = PAD_ID,
)

model = GuppyLM(**config).to(device)

def count_params(m):
    seen = set()
    n = 0
    for p in m.parameters():
        if p.data_ptr() in seen:
            continue
        seen.add(p.data_ptr())
        n += p.numel()
    return n

n_params = count_params(model)
print(f'n_params = {n_params/1e6:.2f} M (target ~8.7 M)')

n_params = 8.08 M (target ~8.7 M)


## 学習ループ

AdamW (`lr=3e-4`, `betas=(0.9, 0.95)`, `weight_decay=0.1`)、線形 warmup 200 ステップ + cosine decay、`max_steps=10000`、`batch_size=32`、勾配クリップ 1.0、混合精度 (AMP) で回す

Colab T4 / L4 などであれば数分〜十数分で完了する

In [9]:
MAX_STEPS   = 10000
WARMUP      = 200
LR          = 3e-4
MIN_LR      = 3e-5
GRAD_CLIP   = 1.0
LOG_EVERY   = 100

use_amp = (device == 'cuda')

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.1,
)
# PyTorch 2.4+ の新 API: torch.amp.GradScaler / torch.amp.autocast
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

def lr_at(step):
    if step < WARMUP:
        return LR * step / max(1, WARMUP)
    prog = (step - WARMUP) / max(1, MAX_STEPS - WARMUP)
    prog = min(1.0, max(0.0, prog))
    coef = 0.5 * (1 + math.cos(math.pi * prog))
    return MIN_LR + (LR - MIN_LR) * coef

def endless(loader):
    while True:
        for batch in loader:
            yield batch

it = endless(train_loader)
model.train()
t0 = time.time()
running = 0.0
for step in range(1, MAX_STEPS + 1):
    lr_now = lr_at(step)
    for g in optimizer.param_groups:
        g['lr'] = lr_now
    x, y = next(it)
    x = x.to(device, non_blocking=True)
    y = y.to(device, non_blocking=True)
    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', enabled=use_amp):
        _, loss = model(x, y)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    running += loss.item()
    if step % LOG_EVERY == 0:
        avg = running / LOG_EVERY
        running = 0.0
        dt = time.time() - t0
        print(f'step {step:5d} | lr {lr_now:.2e} | loss {avg:.4f} | ppl {math.exp(min(20, avg)):.2f} | {dt:.1f}s')


step   100 | lr 1.50e-04 | loss 5.3766 | ppl 216.29 | 5.9s
step   200 | lr 3.00e-04 | loss 3.2035 | ppl 24.62 | 10.1s
step   300 | lr 3.00e-04 | loss 2.5211 | ppl 12.44 | 14.5s
step   400 | lr 3.00e-04 | loss 2.0644 | ppl 7.88 | 18.9s
step   500 | lr 2.99e-04 | loss 1.7290 | ppl 5.63 | 23.1s
step   600 | lr 2.99e-04 | loss 1.4492 | ppl 4.26 | 27.6s
step   700 | lr 2.98e-04 | loss 1.2262 | ppl 3.41 | 31.9s
step   800 | lr 2.98e-04 | loss 1.0510 | ppl 2.86 | 36.2s
step   900 | lr 2.97e-04 | loss 0.9049 | ppl 2.47 | 40.7s
step  1000 | lr 2.96e-04 | loss 0.8052 | ppl 2.24 | 45.1s
step  1100 | lr 2.94e-04 | loss 0.7351 | ppl 2.09 | 49.4s
step  1200 | lr 2.93e-04 | loss 0.6731 | ppl 1.96 | 53.9s
step  1300 | lr 2.92e-04 | loss 0.6266 | ppl 1.87 | 58.3s
step  1400 | lr 2.90e-04 | loss 0.5838 | ppl 1.79 | 62.6s
step  1500 | lr 2.88e-04 | loss 0.5694 | ppl 1.77 | 67.3s
step  1600 | lr 2.87e-04 | loss 0.5498 | ppl 1.73 | 71.7s
step  1700 | lr 2.85e-04 | loss 0.5333 | ppl 1.70 | 76.2s
step  1800 

## 評価

学習データから無作為に数十バッチ取って平均損失と perplexity を確認する

本来は held-out セットを切るべき

- held-out セットを切るというのは、機械学習の標準手法では訓練データ（train）と検証データ（validation / held-out）を分ける原則のこと
  - これは、過学習（overfitting）を検出し汎化性能（generalization）を評価するため
  - 本来は評価用に未使用データ利用するべき

- ここでは、データセットが小さく、モデルも軽量、分割すると訓練データが不足することから、データが少なくて分割が非効率となるため

GuppyLM 公式もこの規模では in-sample の loss 推移を主要指標としている

- in-sample、つまり訓練データを用いて誤差関数を求めて、学習中のlossが下がっているか発散していないかを見る
  - 「訓練データ上の誤差の減り方で学習状況を判断」

- GuppyLM の公式方針として、小規模モデル・小規模データでは厳密な汎化評価よりも、「まず学習が成立しているか」を重視するという観点が述べられている

In [10]:
@torch.no_grad()
def estimate_loss(n_batches=50):
    model.eval()
    losses = []
    it2 = iter(train_loader)
    for _ in range(n_batches):
        try:
            x, y = next(it2)
        except StopIteration:
            break
        x, y = x.to(device), y.to(device)
        _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return sum(losses) / max(1, len(losses))

mean_loss = estimate_loss(50)
print(f'mean loss = {mean_loss:.4f} | ppl = {math.exp(min(20, mean_loss)):.2f}')

mean loss = 0.3755 | ppl = 1.46


## `chat_completion` の再現

GuppyLM 公式 `guppylm/inference.py` の `_format_prompt` と `chat_completion` をそのまま Python 関数として再現する

ユーザ発話を受け取り、ChatML で整形し、`<|im_start|>assistant\n` までを prefix として与え、`<|im_end|>` が出るか最大長に達するまで自己回帰生成する

In [11]:
def _format_prompt(user_message: str) -> str:
    return (
        f'<|im_start|>user\n{user_message}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )

def chat_completion(user_message, max_new_tokens=96, temperature=0.8, top_k=40):
    prompt = _format_prompt(user_message)
    ids = tokenizer.encode(prompt).ids
    ids = ids[-model.max_seq_len:]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(
        idx,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        eos_id=EOS_ID,
    )
    gen_ids = out[0, len(ids):].tolist()
    if EOS_ID in gen_ids:
        gen_ids = gen_ids[: gen_ids.index(EOS_ID)]
    text = tokenizer.decode(gen_ids)
    return text.strip()

## サンプル対話

いくつかのプロンプトで動作を確認する

8.7M という極小サイズなので文法や事実の精度は期待できないが、ChatML 形式の応答構造やグッピー人格の断片が現れれば成功である

In [12]:
for q in [
    'Hello! Who are you?',
    'What do you eat?',
    'Tell me about yourself.',
]:
    print('USER :', q)
    print('GUPPY:', chat_completion(q))
    print('-' * 40)

USER : Hello! Who are you?
GUPPY: i don't know what social media is. i only know fish things.
----------------------------------------
USER : What do you eat?
GUPPY: i have no idea what an alarm clock means. i am a fish.
----------------------------------------
USER : Tell me about yourself.
GUPPY: i don't know what an app is. my brain is the size of a seed.
----------------------------------------


### ipywidgets によるチャット UI

Colab / Jupyter 上で動くシンプルなチャット UI を用意する

入力欄にテキストを書いて Enter または送信ボタンで `chat_completion` を呼び出し、履歴を表示する

In [13]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    log = widgets.Output(layout={'border': '1px solid #888', 'height': '260px', 'overflow': 'auto'})
    inp = widgets.Text(placeholder='Say something to the guppy...', layout={'width': '70%'})
    btn = widgets.Button(description='Send', button_style='primary')

    def on_send(_=None):
        msg = inp.value.strip()
        if not msg:
            return
        inp.value = ''
        with log:
            print(f'USER : {msg}')
            reply = chat_completion(msg)
            print(f'GUPPY: {reply}')
            print('-' * 40)

    btn.on_click(on_send)
    inp.on_submit(on_send)
    display(widgets.VBox([log, widgets.HBox([inp, btn])]))
except Exception as e:
    print('ipywidgets UI is not available in this environment:', e)

### CUI フォールバック

ipywidgets が使えない環境(通常の `python` REPL や一部の Jupyter ランタイム)向けに、`input()` ベースの簡易ループも用意しておく。空行または `quit` で終了する

In [14]:
def cui_chat():
    print('GuppyLM CUI chat. Empty line or "quit" to exit.')
    while True:
        try:
            msg = input('USER : ').strip()
        except (EOFError, KeyboardInterrupt):
            print(); break
        if not msg or msg.lower() == 'quit':
            break
        print('GUPPY:', chat_completion(msg))

# 対話したいときは次行のコメントを外す。
# cui_chat()

# LLM 効率化手法 ― スケールの壁を越える技術群

本章で実装した GuppyLM は 8.7M パラメータの極小モデルだが、同じ Transformer アーキテクチャが数百 B パラメータに及ぶ現代の LLM を支えている

規模が 4〜5 桁大きくなると、訓練・推論の計算コスト、メモリ消費、レイテンシが実用上の深刻な壁となる
- 必要なリソースが増えると、トークン規模の制限や、巨大モデルが利用できないこと、消費電力や計算資源の要求量が増えるためコストが増大することなどから、LLMサービス提供企業競争で不利となる

本セクションでは、この壁を越えるために研究・実用化されてきた効率化手法を体系的に整理する

大きく以下の 8 カテゴリに分類できる

- **Attention の効率化** ― Sparse / Linear / RNN / SSM による計算量削減
- **KV キャッシュ削減** ― 推論時メモリの壁を超える
- **HW・システム最適化** ― アーキテクチャを変えずに実装レベルで高速化
- **モデル圧縮** ― 量子化・プルーニング・知識蒸留で既存モデルを小さく軽くする
- **MoE（Mixture of Experts）** ― 疎な活性化で計算効率を上げる
- **LoRA 系 PEFT** ― ファインチューニング時のパラメータ・メモリ削減
- **コンテキスト長拡張** ― 既存モデルの文脈窓を事後的に広げる
- **ハイブリッドアーキテクチャ** ― Attention と代替機構の融合

## Attention の効率化 ― 計算量削減

Transformer の自己注意機構は、系列長 $n$ に対して計算量・メモリともに $O(n^2)$ のスケーリングを示す
長文書の処理や長い生成タスクでは、この二乗コストが実用上の障壁となる
以下では、この問題に対するアプローチを体系的に整理する

### Sparse Attention ― 疎な注意パターンで計算を削減

**動作原理**

標準的な Self-Attention では、トークン $i$ が系列中の全トークン $j$ に対して注意を計算する

しかし現実の言語では、遠距離の全トークンが等しく重要なわけではない
**Sparse Attention** はこの観察を活かし、あらかじめ決めた「注意すべきトークン集合」だけに計算を限定する

注意パターンの典型的な構造は以下の組み合わせで設計される：

- **Local window（局所窓）**: 各トークンが近傍 $w$ トークンのみに注目 → 文脈の局所性を捉える
- **Global tokens（グローバルトークン）**: 特定のトークン（例: `[CLS]`）が全系列に双方向で注意 → 大域情報の集約
- **Random blocks（ランダムブロック）**: ブロック単位でランダムに選んだ遠距離トークンへの注意 → 長距離依存のカバー

**Longformer（Allen AI, 2020）** は `sliding window + global tokens` の構成を採用
- 局所窓の計算量は $O(n \cdot w)$ となり、$w \ll n$ の場合に大幅な削減が得られる
- 文書分類タスクでは `[CLS]` トークンをグローバルとして機能させることで、全体文脈を保持しながら長文を処理できる

**BigBird（Google, 2020）** は `random + local + global` の三要素を組み合わせたブロック単位の Sparse Attention を提案
- 理論的には Sparse Attention が完全な Turing 完全性を保持できることを証明しており、単なる近似ではなく表現力の保証を与えた点が重要な貢献

**2025年の発展**

- **DeepSeek Sparse Attention（DSA）**: "lightning indexer" と呼ばれる動的トークン選択機構を導入し、注意コストを従来比 50% 以上削減した。静的なパターンではなく、入力ごとに重要トークンをオンザフライで選択する点が特徴的である
- **MoSA（Mixture of Sparse Attention）**: 各注意ヘッドに独立したルーターを学習させ、ヘッドごとに異なる疎化パターンを自動選択する。Mixture-of-Experts の考え方を Attention の疎化に適用した設計といえる

### Linear Attention ― カーネル近似で $O(n^2)$ を $O(n)$ へ

標準的な Softmax Attention の計算式は以下のように書ける：

$$\text{Attn}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

この `softmax` の存在が $O(n^2)$ の根本原因である
- 行列 $QK^\top$ を明示的に構築しなければならないため

**Linear Attention** はカーネル関数 $\kappa(q, k) = \langle \varphi(q), \varphi(k) \rangle$ による近似を使い、次のように書き換える

$$\text{Attn}(Q, K, V)_i \approx \frac{\varphi(q_i)^\top \left(\sum_j \varphi(k_j) v_j^\top\right)}{\varphi(q_i)^\top \left(\sum_j \varphi(k_j)\right)}$$

- 括弧内の $\sum_j$ は系列全体で一度だけ計算・累積できるため、計算量が $O(n)$ に落ちる

**Performer（Google, 2020）** は `FAVOR+`（Fast Attention Via positive Orthogonal Random features）アルゴリズムを提案した
正規直交乱数基底を使って softmax カーネルを不偏推定し、注意行列を近似する

- 重要な工夫は「正値性（positive features）」の確保で、これにより数値的に安定した近似が得られる点

**トレードオフ**：Linear Attention の本質的な課題は、**softmax の鋭いピーク分布が失われる**ことにある

- Softmax は少数のトークンに注意を集中させる「選択的」な挙動をするが、カーネル近似ではこの鋭さが平滑化される
- 結果として、特定のトークンに強く依存するタスク（固有表現認識など）では精度低下が起きやすい

**2025–2026年の発展**

- **FlashLinearAttention / GLA（Gated Linear Attention）**: ゲート機構を加えて表現力を補いつつ、FlashAttention-2/3 対比で長系列において **2–4倍の高速化** を達成した
- **Log-Linear Attention（ICLR 2026）**: $O(n \log n)$ の計算量で softmax Attention に近い精度を実現し、Linear Attention と標準 Attention の中間点を埋める位置づけとなる

### RWKV 系 ― RNN への回帰

**背景：Attention Free Transformer（AFT）**

2021年に提案された **AFT（Attention Free Transformer）** は、$Q \cdot K$ の内積を完全に廃止した

代わりに各トークンの重みを学習可能なバイアスで与える設計だが、これが「RNN 的な逐次処理への回帰」という方向性を示す先駆けとなった

**RWKV-4（2023）**

`RWKV`（Receptance Weighted Key Value）は、時間方向の混合（time-mixing）とチャネル方向の混合（channel-mixing）の二層構造で構成される

- Time-mixing ブロックは本質的に RNN の隠れ状態更新に相当し、次の加重線形補間（lerp）の形で状態を更新する

$$\text{new_state} = \text{lerp}(\text{old_state},\ g_{\text{input}} \times v_{\text{new}},\ w_{\text{decay}})$$

- $w_{\text{decay}}$ は学習可能な減衰係数であり、過去情報をどの程度保持するかを制御する

**Eagle / RWKV-5（2024）** は隠れ状態をスカラーベクトルから**行列値（matrix-valued state）**に拡張し、マルチヘッド構造を導入した
最大 7.5B パラメータのモデルで、Transformer に匹敵する性能を示した

**Finch / RWKV-6（2024）** はデータ依存の動的な時間減衰（dynamic data-dependent recurrence）を採用し、入力ごとに $w_{\text{decay}}$ が変化するようになった
- 14B モデルまでスケールアップされ、多言語ベンチマークで競争力のある結果を示している

**RWKV-7 "Goose"（2025年3月）** は **Dynamic State Evolution** と呼ばれる新機構を導入し、状態遷移行列 $A$ を完全にデータ依存にした
- 理論的には `TC₀`（並列計算クラス）の表現力上限を超えることが示されており、固定構造の RNN では解けなかった問題クラスへの対応が可能になった

**RWKV 系の核心的利点**

| 項目 | Transformer | RWKV |
|------|-------------|------|
| 学習時計算量 | $O(n^2)$（並列可能） | $O(n)$（並列可能） |
| 推論時メモリ | $O(n)$ KV キャッシュ | $O(1)$ 固定サイズ状態 |
| 長系列への外挿 | 位置エンコーディング依存 | 自然に外挿可能 |

推論時の KV キャッシュが不要で **$O(1)$ のメモリ** で動作する点は、組み込みデバイスやストリーミング推論で特に重要な強みである

### SSM 系 ― 状態空間モデル

**理論的起源：制御理論の状態方程式**

SSM（State Space Model）は制御工学の古典的な状態方程式に由来する：

$$h'(t) = A h(t) + B x(t), \quad y(t) = C h(t) + D x(t)$$

- $h(t)$ は隠れ状態、$x(t)$ は入力、$y(t)$ は出力を表す
- この連続時間系を離散化してニューラルネットワークの層として組み込むのが SSM 系モデルの基本思想である

**S4（Gu et al., 2021）** の最大の貢献は **HiPPO 初期化** にある

- HiPPO（High-order Polynomial Projection Operators）理論に基づいて行列 $A$ を初期化することで、過去の信号を多項式基底で効率的に記憶できる

- 学習は畳み込みとして $O(n \log n)$ で並列実行でき、推論時は RNN として $O(1)$ で動作するという**二重の効率性**を実現した

**Mamba（Gu & Dao, 2023）** は S4 の弱点（$A, B, C$ が入力によらず固定）を克服した `Selective SSM` である

- これらのパラメータを入力 $x$ に応じて動的にゲーティングする：

$$B = f_B(x), \quad C = f_C(x), \quad \Delta = f_\Delta(x)$$

- このデータ依存性により、**必要な情報だけを選択的に状態に保持**し不要な情報を積極的に忘れることができる
- Transformer に匹敵する言語モデリング性能を示しながら、**推論速度は 5–10 倍高速**という結果を報告した

**Mamba-2（2024）** は `SSD`（State Space Duality）層を導入し、SSM と注意機構の数学的等価性を明示化した
- この統一的視点により実装を大幅に簡略化し、**Mamba 比で 2–8 倍の高速化**を達成した

**Mamba-3（2026）** は推論効率を第一設計原則として構築された最新世代で、複雑な動態（complex dynamics）と MIMO（Multiple Input Multiple Output）構造を採用し、**Mamba-2 と同等のパープレキシティを状態サイズ半分で実現**した

## KV キャッシュ削減 ― 推論時メモリの壁を超える

### KV キャッシュとは何か

自己回帰的なテキスト生成では、モデルは1トークンずつ出力を生成する
各ステップで新しいトークンは過去の全トークンに注意を計算する必要があるが、過去トークンの **Key（K）行列と Value（V）行列** は変化しない
- この不変な K, V を再計算のたびに捨てずに保存しておくのが **KV キャッシュ** である

キャッシュのメモリ量は以下の積で決まる：

$$\text{KV キャッシュサイズ} = 2 \times n_\text{layers} \times n_\text{heads} \times d_\text{head} \times L \times \text{dtype\_bytes}$$

ここで $L$ は系列長である

- たとえば **70B モデルで 128K コンテキスト**の場合、KV キャッシュだけで **40GB 超**に達することがある

- これはモデルの重みとは別に必要なメモリであり、長文書処理や大規模バッチ推論の実用上の障壁となる

### KV ヘッド共有

KV キャッシュの削減に向けた最もシンプルかつ広く普及したアプローチが、**ヘッド数の共有**である

**MHA（Multi-Head Attention）** では、各クエリヘッド $i$ が固有の $K_i, V_i$ を持つ

- 表現力は最大だが、ヘッド数 $H$ に比例してキャッシュも増大する

**MQA（Multi-Query Attention, Shazeer 2019）** は全 $H$ 個のクエリヘッドが **単一の K, V ペアを共有**する

- キャッシュ量は MHA の $1/H$ となり劇的な削減が得られるが、全ヘッドが同じ K, V を参照することで**表現の多様性が失われ**、特定タスクで品質低下が報告されている

**GQA（Grouped-Query Attention, Ainslie et al. 2023）** は MHA と MQA の中間として、$H$ 個のクエリヘッドを $G$ グループに分割し、グループ内で K, V を共有する

- $G = 1$ なら MQA、$G = H$ なら MHA と等価になり、$G$ の選択で**品質と効率のトレードオフを連続的に制御**できる

**Llama 3, GPT-4, Claude** などの主要なプロダクションモデルで採用されており、事実上の標準設計になりつつある

**MLA（Multi-Head Latent Attention, DeepSeek-V2/V3, 2024–2025）** はより根本的なアプローチとして、K, V を**低ランクの潜在空間に圧縮**する設計を提案した

- 通常の K, V の代わりに、低次元の潜在ベクトル $c_{KV} \in \mathbb{R}^{d_c}$（ただし $d_c \ll d_\text{model}$）だけをキャッシュし、推論時に次のように展開して使用

$$K = W^K c_{KV}, \quad V = W^V c_{KV}$$

- この設計により以下を達成

  - **KV キャッシュを MHA 比 93.3% 削減**
  - **スループット 5.76 倍向上**（DeepSeek-V2 報告値）

- ヘッド数を減らすのではなく**情報の次元を圧縮する**という発想の転換が鍵であり、各ヘッドの表現の多様性を犠牲にしにくい点で GQA より有利な場合がある

### KV キャッシュ圧縮・選択的保持

ヘッド共有とは異なる方向として、**生成が進む中でキャッシュ自体を動的に削減・選別**するアプローチがある

**H2O（Heavy Hitter Oracle, Zhang et al. 2023）** は注意機構において少数の「重要トークン」（Heavy Hitters）が累積注意スコアの大半を占めるという観察を活かし、**各層のキャッシュを固定バジェット内に収める**ために以下のルールで保持するトークンを選択する：

1. 直近の $k_\text{recent}$ トークン（局所文脈として常に保持）
2. 累積注意スコアが高い上位 $k_\text{heavy}$ トークン（Heavy Hitters）

追加の再学習なしに推論時のみで適用できるため、既存モデルへの後付けが容易である

**SnapKV（2024）** は H2O が**生成全体を通じた累積スコア**を使うのに対し、**プロンプト（入力指示）ウィンドウ内の局所注意パターン**に注目する

- 入力処理段階で重要な KV 位置をクラスタリングし、その後の生成全体でそのスナップショットを使い回す

- 長いシステムプロンプトや文書を含む RAG（Retrieval-Augmented Generation）シナリオで特に効果的である

**PyramidKV（2024）** は経験的観察として、**Transformer の下位層（入力近傍）は注意が広く分散**し、**上位層は特定トークンに鋭く集中
**する傾向があることを利用する

- 下位層ほど多くのキャッシュを割り当て、上位層では少なく抑えるピラミッド型のバジェット配分により、均等配分と比べて同じ総バジェットでより多くの重要情報を保持でき、長系列での精度低下を抑制する

**DeltaKV（2025）** は生成が進むにつれて遠距離の K, V ベクトルが隣接ステップ間で変化が小さくなる傾向（残差的な類似性）を利用し、フルベクトルの代わりに**前ステップからの差分（デルタ）**だけを保存する残差圧縮方式を採用している

> **補足** ― PagedAttention などのシステム的 KV キャッシュ管理（「何を保持するか」ではなく「どのようにメモリを管理するか」）は、次のシステム最適化セクションで扱う

## HW・システム最適化 ― アーキテクチャを変えずに高速化する

GPU のアーキテクチャやサービング基盤を工夫することで、モデルの重みやアーキテクチャを一切変えずに大幅な高速化・省メモリ化を実現できる

### I/O 最適化 ― FlashAttention

GPU には二種類のメモリが存在する

- **HBM (High Bandwidth Memory)**: 容量大（数十 GB）、帯域幅は広いが、SRAM に比べると遅い
- **SRAM (on-chip memory)**: 容量小（数 MB 程度）、アクセス速度は桁違いに速い

- 標準的な Attention 計算は Q, K, V を HBM から読み込み、$n \times n$ の Attention 行列を HBM に書き出し、再び読み込んで V と掛け合わせる

- シーケンス長 $n$ が大きくなると、この $n \times n$ 行列の読み書きがボトルネックとなる

- 計算量ではなく**メモリ帯域幅**が律速する、いわゆる「memory-bound」な処理になる

**FlashAttention (Dao et al., NeurIPS 2022)** は「$n \times n$ 行列を一度も HBM に書き出さない」という発想で設計された IO-aware アルゴリズムである
- アイデアの核心は**タイリング (tiling)** と **オンライン softmax** の組み合わせで、Q, K, V を小さなブロックに分割して SRAM 上のみで Attention の部分計算を完結させる
- 分母の正規化定数を逐次更新する数値的に安定な手法により、全要素を揃えなくても正確な softmax を計算できる
- 結果として HBM への書き込みは最終出力のみとなり、Attention 行列のメモリ使用量が $O(n^2)$ → **$O(n)$** に改善、A100 GPU 上で標準実装比 **2〜4 倍の高速化**を達成した

**FlashAttention-2 (Dao, 2023)** は GPU 内のスレッドブロックへの作業分割を最適化し、行列積以外の演算を削減することで A100/H100 上でさらに約 **2 倍**の高速化を実現した

**FlashAttention-3 (Shah et al., 2024)** は H100 の Hopper アーキテクチャに特化した非同期パイプライニングと FP8 データ型のサポートにより、ハードウェアのピーク演算性能に近い利用効率を達成した
- FlashAttention は今や PyTorch や主要フレームワークに標準統合されており、「使わない理由がない」基盤技術となっている

### 推論メモリ管理 ― PagedAttention

LLM のサービング（推論）では、各リクエストに対して KV キャッシュ用の**連続したメモリ領域**を事前確保するため、リクエストが最大長まで達しない場合の**内部断片化**が深刻な問題となる
- GPU メモリの最大 70% 以上がこの断片化で失われるケースも報告されている

**PagedAttention (Kwon et al., SOSP 2023)** は OS のページングメモリ管理にヒントを得た手法で、KV キャッシュを固定サイズの「**ページ**」に分割し、論理的なトークン位置から物理ページへのマッピングテーブルを管理する
- 連続したメモリは不要で、空いているページをどこからでも割り当てられるため、**内部断片化がほぼゼロ**になる
- さらにビームサーチや並列サンプリングで同一プレフィックスを持つリクエスト間で**ページを共有**（copy-on-write）でき、バッチサービングのスループットが従来比 **10 倍以上**向上する

**vLLM** は PagedAttention を中心に構築されたオープンソースのサービングエンジンで、業界標準的な地位を確立している

### 推論加速

**Speculative Decoding (Leviathan et al., 2023 / Chen et al., 2023)** は LLM の自己回帰生成が本質的に逐次処理であるという制約を巧妙に回避する
- 小さな**ドラフトモデル**が $k$ トークンを高速に逐次生成し、大きな**ベリファイアモデル**が $k$ トークンを 1 回の forward pass で並列検証する
- 統計的な棄却サンプリングにより、受け入れられたトークン列は大モデル単独の出力と**数学的に同一の分布**に従う
- 品質を一切損なわずに **2〜4 倍の wall-clock 高速化** を実現でき、Google の Gemini や Anthropic の Claude にも採用されている

**SGLang (Zheng et al., 2024〜2025)** は RAG やチャットボットのように多数のリクエストが共通のシステムプロンプトを共有する状況を最適化する

**RadixAttention** により KV 計算結果を Radix 木にキャッシュし、共有プレフィックスを持つ新しいリクエストはキャッシュを再利用して差分のみ計算する
- 典型的なデプロイパターンで vLLM 比 **29% 高速化**を達成し、2026 年時点で **40 万 GPU 以上**のデプロイ実績がある

**TensorRT-LLM (NVIDIA)** は CUDA/Triton カーネルレベルの低レベル最適化を施した推論エンジンで、NVIDIA ハードウェア上で最大スループットを追求する
- コンパイル時間が必要なためイテレーションコストは高いが、大規模本番デプロイで最高性能を求める場合の選択肢となる

### 分散長文脈処理

コンテキスト長 $n$ の KV キャッシュは $O(n)$ のメモリを消費する
- 1M トークンを超えるような長文脈では、単一 GPU の HBM に収まらなくなる

**Ring Attention (Liu et al., ICLR 2024)** はシーケンスを複数 GPU に分散させる手法で、各 GPU はシーケンスの一部（Q ブロック）を担当し、K, V ブロックを**リングトポロジー**で隣の GPU へ順番に転送する
- 各 GPU は受け取った K, V で自分の Q ブロックの Attention を計算しながら転送を重ね、通信と計算を**完全オーバーラップ**させることで通信コストをほぼ隠蔽する
- GPU 数に対してほぼ**線形スケール**し、有効コンテキスト長 = GPU 数 × 1 GPU あたりのコンテキスト長となる
- 2026 年時点で、シングルノードで 1〜2M トークン、分散システムで 10M+ トークンが実用域に入っている

## モデル圧縮 ― 既存モデルを小さく軽くする

学習済みモデルをそのまま小さくする手法群で、エッジデバイスへのデプロイや推論コスト削減に不可欠な技術である

### 量子化 (Quantization)

ニューラルネットワークの重みは通常 FP16 または BF16（16 ビット浮動小数点）で保存される

量子化はこれを INT8（8 ビット整数）や INT4（4 ビット整数）などより低精度の表現に変換することで、モデルサイズの削減（FP16 → INT4 で **4 倍**）、メモリ帯域幅の削減による推論速度向上、整数演算によるハードウェアレベルでの高速化を実現する

**Post-Training Quantization (PTQ)** として、再学習なしで学習済みモデルを量子化する手法群が発展している

**GPTQ (Frantar et al., 2023)** は `OBQ (Optimal Brain Quantization)` フレームワークを LLM に適用した手法で、各レイヤーについて量子化誤差を最小化する重みの丸め方を Hessian 行列の近似を用いて計算する

- INT4 精度でも品質劣化を最小限に抑え、`Marlin` カーネルによる GPU 最適化実装で実際の推論スループット向上も実現する

**AWQ (Lin et al., MLSys 2024)** は「どの重みが重要か」を事前に分析するアプローチを取る

- 全重みの約 **1%** が「高活性化チャネル」に対応し品質への影響が不均衡に大きいという観察に基づき、この顕著な重み（salient weights）を保護しつつ残りを積極的に量子化する

- インストラクションチューニングモデルやマルチモーダルモデルで GPTQ を上回る品質を示す

**AQLM (Yandex, 2024)** は複数コードブックを使う**ベクトル量子化**手法で、複数の重みをまとめてベクトルとして符号化しコードブックのインデックスで表現する

**3 ビット未満**の実効圧縮率を達成しつつ品質を維持し、3 ビット以下の圧縮では現在の Pareto 最良フロンティアを形成する

**QuIP# (Cornell, 2024)** は **Hadamard 変換による非整合化処理（incoherence processing）**で重み行列を回転させ、量子化誤差が特定方向に集中しないよう前処理する

- ラティスコードブックと組み合わせることで、**2 ビット量子化でほぼ無損失**に近い品質を達成する

**TurboQuant (Google, 2026年3月)** は重み量子化ではなく **KV キャッシュの量子化**を対象とした手法で、長文脈シナリオでは KV キャッシュがメモリを支配するため、ここを圧縮する意義が大きい

- Hadamard 回転による KV キャッシュの前処理により **6 倍の KV キャッシュ圧縮**と H100 上での Attention 計算 **8 倍高速化**を達成し、長文脈 LLM にとってゲームチェンジャーとなりうる技術である

**フォーマットの使い分け**

| フォーマット | 用途 |
|---|---|
| `GGUF` | CPU・エッジデバイスでの推論（llama.cpp） |
| `GPTQ` | GPU 上でのバッチ推論スループット最大化 |
| `AWQ` | インスト���クションチューニング・マルチモーダルモデル |

### プルーニング (Pruning)

学習済みモデルの冗長な重みを除去することでモデルを小さく・速くする手法で、除去の粒度によって二種類に分かれる

**非構造化プルーニング** は個々の重みをゼロにする最も細粒度な手法である

**SparseGPT (Frantar & Alistarh, ICML 2023)** はプルーニングをキャリブレーションデータ上のスパース回帰問題として定式化し、各重みを除去した際の出力誤差を Hessian 近似で補正する

- 175B パラメータのモデルでも **50% 以上のスパーシティ**を精度劣化最小限で達成し、再学習も不要である

**Wanda (Sun et al., 2024)** は `|重みの値| × 対応する活性化の大きさ` という単純なスコア基準でプルーニングする

- Hessian 計算不要で**計算コストが SparseGPT の数分の一**ながら、品質は SparseGPT に匹敵する

- なぜこのスコアが機能するのか：重みが小さくても乗算される活性化が大きければ出力への寄与は大きく、逆に活性化が小さい経路の重みは除去しても影響が少ないため

**構造化プルーニング** はレイヤー・Attention ヘッド・隠れ次元全体を削除するより粗粒度な手法で、汎用ハードウェアでそのまま高速化できる

**Minitron (NVIDIA, 2024)** は深さ方向と幅方向のプルーニングを組み合わせ、知識蒸留で品質を回復する手法で、Llama 3.1 8B → **4B**、Mistral NeMo 12B → **8B** へのスリム化を実現した

**ShortGPT (2024)** は各レイヤーの出力を Block Influence スコアで評価し、スコアが低いレイヤー（入力をほぼそのまま通過させているだけのレイヤー）を除去する

### 知識蒸留 (Knowledge Distillation)

大きな**教師モデル（teacher）**の知識を小さな**生徒モデル（student）**に移す学習パラダイムである

- 通常の学習ではラベル（正解クラスのみ 1、他は 0）から学ぶが、蒸留では教師モデルの**ソフトラベル（ロジット分布全体）**から学ぶ

- ソフトラベルには教師モデルが学習した**クラス間の類似関係**が反映されており、ハードラベルより遥かに情報量が大きい

- LLM への蒸留には主に二つのアプローチがある：

  - **ロジット蒸留**: 教師モデルの出力トークン分布を教師信号として直接使用
  - **合成データ蒸留**: 教師モデルに大量のテキストを生成させ、それを学習データとして使用（ブラックボックスの教師にも適用可能）

- プルーニングとの組み合わせ（Minitron アプローチ）として「まず構造化プルーニングで小さなモデルを作り、次に蒸留で品質を回復する」パイプラインが現在の標準的なアプローチとなっている

蒸留は大手 AI 企業の商用モデルファミリーで標準手法となっており（OpenAI: GPT-4 → GPT-4o mini、Anthropic: Claude Opus → Claude Haiku、Google: Gemini Ultra → Gemini Flash）、「大きなモデルを作って、蒸留で小さく展開する」パターンが業界全体のデファクトスタンダードとなりつつある

## MoE（Mixture of Experts）― 疎な活性化で計算効率を上げる

### MoE の基本原理

大規模言語モデルを「賢く」するためにはパラメータ数を増やすことが効果的だが、パラメータ数に比例して計算コストも増えると推論コストが現実的でなくなる

**MoE** はこの矛盾を解決するアーキテクチャで、「総パラメータ数は多いが、1 トークンあたりの計算量は少ない」という状態を実現する

核心的なアイデアは、Transformer の各層に複数の**エキスパート（expert）**サブネットワーク（多くの実装では FFN を複数用意）を持たせ、**ルーター（router / gating network）**と呼ばれる軽量なネットワークが各トークンに対して**上位 $k$ 個のエキスパート（top-k routing）**を選択すること

数式で表すと、MoE 層の出力 $y$ は次のように書ける：

$$y = \sum_{i=1}^{N} g_i(x) \cdot E_i(x)$$

- ここで $E_i(x)$ は $i$ 番目のエキスパートの出力、$g_i(x)$ はルーターが割り当てるゲーティング係数で、選ばれた上位 $k$ 個以外の係数はゼロになる
- これにより「定義上は全エキスパートが存在するが、実際に計算されるのは $k$ 個分だけ」という**疎な活性化（sparse activation）**が実現される

実例として **Mixtral 8x7B**（Mistral, 2023）では、各層に 8 つのエキスパートを持ち各トークンに top-2（$k=2$）を選択する
総パラメータ数は **46.7B** だが 1 forward pass で活性化されるパラメータは **12.9B** のみで、「7B モデル程度の計算コストで 47B モデルに近い表現力」を持つ

### ロードバランシングの課題

MoE の最大の落とし穴は**ルーターが特定のエキスパートに偏る**ことで、他のエキスパートが「メモリを占有するだけの使われないパラメータ」になる **expert collapse（エキスパート崩壊）**が起きうる

対策として以下のアプローチが提案されている：

- **補助損失（auxiliary balance loss）**: 各エキスパートへのルーティング比率が均等になるよう正則化項を訓練損失に加える
- **容量係数（capacity factor）**: 1 エキスパートが 1 バッチで受け取れるトークン数に上限を設ける
- **Noisy Top-K Gating**: ルーターの出力にノイズを加えて探索を促し、過度な集中を防ぐ

### MoE アーキテクチャの系譜

**GShard（Lepikhin et al., Google, 2020）** は機械翻訳タスクに MoE を適用した初期の大規模実装で、600B パラメータを複数 TPU にエキスパートを分散配置する設計を示した

**Switch Transformer（Fedus et al., Google, 2021）** は $k=1$（各トークンにエキスパートを 1 つだけ割り当て）に単純化することで通信オーバーヘッドを削減し、**1.6T パラメータ**までスケールさせた
「$k$ を増やすほど良いわけではない」という重要な知見を示した論文でもある

**Mixtral 8x7B（Mistral AI, 2023）** は MoE を一般公開モデルに持ち込み、同サイズの dense モデル（Llama 2 13B など）を多くのベンチマークで上回りながら推論コストを大幅に抑えた
現在もオープンソース MoE のデファクトスタンダードとして広く使われている

**DeepSeek-MoE（2024）** は「細粒度エキスパート（fine-grained experts）」として 1 層あたり 16〜64 の細かいエキスパートを用意し、さらに全トークンに対して常に活性化される「**共有エキスパート（shared experts）**」を追加した
- 共有エキスパートが汎用的な知識を担当し、専門的なエキスパートは特定パターンへの特化に集中できるため、ロードバランシングが改善される

**DeepSeek-V3（2025）** は MoE の現時点での到達点を示す
総パラメータ **671B**、1 層あたりエキスパート数 **256**（Mixtral の 8 と比較して桁違いに多い）で、**MLA** と組み合わせて KV キャッシュ効率も最適化し、訓練コストあたりのパープレキシティで当時の最高水準を達成した

**DeepSeek-R1（2025）** はこの MoE 基盤を推論（reasoning）特化にチューニングしたモデルで、強化学習によって「考えてから答える」能力を強化している

### MoE の利点と課題

**利点**：総パラメータ数（= 記憶できる知識の量）を増やしても 1 トークンあたりの FLOP はほぼ一定に保てる
- 異なるエキスパートが異なる言語・ドメイン・構文パターンを専門的に処理するよう自然に分業が生まれることも実験的に観察されている

**課題**：分散訓練時の all-to-all 通信がネットワーク帯域のボトルネックになること、推論時も全エキスパートの重みをメモリに置く必要があること（671B の DeepSeek-V3 を動かすには数十枚の GPU が必要）が挙げられる

**2025〜2026 年のトレンドとして、新規公開されるオープンソース大規模モデルの 60% 以上が MoE アーキテクチャを採用しており、「大型モデル = MoE」が業界の常識になりつつある**

## LoRA 系 ― パラメータ効率的ファインチューニング（PEFT）

### なぜ PEFT が必要か

事前学習済み LLM を特定タスクに適応させるにはファインチューニングが必要だが、**70B パラメータのモデルを全パラメータ更新（full fine-tuning）**しようとすると深刻な問題が生じる
- 70B パラメータ × 16-bit = 約 **140GB** の重みストレージに加え、Adam オプティマイザの 1st/2nd moment でさらに **280GB** が追加され、合計 **420GB 以上**の GPUメモリが必要になる
- 多くの下流タスク（例えば「法律文書の要約」「特定企業のトーンで応答する」）は、モデルが持つ全知識を書き換える必要はなく**ごく一部の振る舞いを調整する**だけで足りる

**PEFT（Parameter-Efficient Fine-Tuning）**は、ベースモデルの重みを**凍結（freeze）**したまま少数の追加・修正パラメータだけを訓練する手法群で、「変えなくていいものは変えない」という発想でメモリと計算を節約する

### LoRA（Low-Rank Adaptation）

**LoRA**（Hu et al., Microsoft, 2021）は PEFT の中で最も広く普及した手法で、「大規模モデルのファインチューニング時、重み行列の**更新量 $\Delta W$ は低ランク（low-rank）構造を持つ**」という特徴を利用する

通常、重み行列 $W \in \mathbb{R}^{d \times d}$ の更新 $\Delta W$ はフルランクになり得るが、実験的に「タスク適応に必要な変化は内在的に低次元だ」ということが示されている
- LoRA はこれを利用し、$\Delta W$ を 2 つの低ランク行列の積に分解する：

$$\Delta W = BA, \quad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times d},\quad r \ll d$$

ランク $r$ は通常 8〜64 に設定する
- 例えば $d = 4096$、$r = 16$ の場合、フル更新のパラメータ数 $4096^2 \approx 1677$ 万に対し、LoRAのパラメータ数は $16 \times (4096 + 4096) \approx 13$ 万で**削減率 99.2%** となる

訓練時は $W$ を凍結し $A$ と $B$ だけを更新する
- $A$ はランダム初期化、$B$ はゼロ初期化することで訓練開始時の $\Delta W = BA = 0$（元モデルと同じ状態）が保証される

**推論時**は $W' = W + BA$ として事前計算でマージでき、**追加の推論オーバーヘッドはゼロ**である

LoRA は主に Attention 層の **Q, K, V, O 射影行列**に適用され、FFN 層に適用することもある

### LoRA の発展形

**QLoRA（Dettmers et al., 2023）** は LoRA に量子化を組み合わせた手法で、ベースモデルを **4-bit NF4（NormalFloat4）量子化**でロードし、LoRA アダプタだけを **BF16** で訓練する
- 効果は劇的で、**65B モデルを 48GB GPU 1 枚**でファインチューニング可能にし（以前は数十枚の GPU が必要）、16-bit LoRA と比較してメモリを約 **75% 削減**しながら精度は full fine-tuning の 99% 以上を維持する

**DoRA（Liu et al., NVIDIA, ICML 2024 oral）** は重み行列を**マグニチュード（magnitude）× 方向（direction）**に分解するという新しい視点を導入した
- $$W = m \cdot \frac{V}{\|V\|_c}, \quad m \in \mathbb{R}^{1 \times d},\ V \in \mathbb{R}^{d \times d}$$
- 「方向成分 $V$ に LoRA を適用し、マグニチュード $m$ は個別に学習する」方針により、フルファインチューニングの学習ダイナミクスに近い挙動を実現し、**LoRA より少ないランクで同等の精度**に到達しやすい

**QDoRA（Answer.AI, 2024）** は DoRA + 4-bit 量子化の組み合わせで、QLoRA 同等のメモリ効率を保ちつつ DoRA の収束改善を得る

**GaLore（Zhao et al., 2024）** は LoRA の「重み更新を低ランクにする」アプローチとは異なり、**勾配を低ランク部分空間に射影**する手法である
- ベースモデルのパラメータを直接更新しながらも勾配の計算・保存を低ランク空間で行うことで、**フルランクの重み更新能力を持ちつつ LoRA 同等のメモリ削減**を実現する

**Adapter 系の手法群**も PEFT の重要なカテゴリである：
- **Prefix Tuning**（Li & Liang, 2021）: 各 Transformer 層の Key・Value に学習可能な「プレフィッ���ス」ベクトルを追加
- **Prompt Tuning**（Lester et al., 2021）: 入力シーケンスの先頭に学習可能なソフトトークンを付加。モデル規模が大きいほどフルファインチューニングに近い性能を発揮する
- **IA³**（Liu et al., 2022）: Key、Value、FFN の活性化にスケーリングベクトルを掛けるだけというシンプルな手法で、学習パラメータ数が極めて少ない

### PEFT の現状（2025〜2026）

QLoRA や DoRA を適切に設定したファインチューニングは、同データでの full fine-tuning の **95% 以上の精度**を多くのタスクで達成する
訓練パラメータ数 **1% 未満**、メモリ削減 **10〜20 倍**が標準的な効率水準であり、Hugging Face の `peft` ライブラリや `trl` ライブラリにより数行のコードで利用可能である

標準的なワークフローとして、リソース制約がある場合は QLoRA、品質最優先の場合は BF16 フルランク LoRA または DoRA、大規模適応では GaLore またはフルファインチューニングが選択される

## コンテキスト長拡張 ― 既存モデルの文脈窓を事後的に広げる

### 位置符号化の復習

Transformer は本質的に**順列不変（permutation-invariant）**なアーキテクチャであり、位置情報を明示的に与えなければモデルは語順を認識できない
- GuppyLM では最もシンプルな**学習可能絶対位置埋め込み**を採用した
これは最大系列長 `max_len` 分の埋め込みベクトルをパラメータとして学習する方式で、実装は単純明快だが「学習した長さを超えた位置には対応できない」という根本的な制約がある
- この問題を解決するために提案された代表的な手法が **RoPE（Rotary Position Embedding, Su et al., 2021）**である

RoPE のアイデアは「絶対位置を直接埋め込むのではなく、Q・K ベクトルを位置に応じた角度で回転させることで相対距離を暗黙的に表現する」というもので、埋め込み次元を 2 次元のペアに分割し各ペアに回転行列を適用する：

```
Q_rotated[m] = R(m·θ) · Q[m]
K_rotated[n] = R(n·θ) · K[n]
```

- ここで `θ` は次元ごとに異なる周波数パラメータで、Q と K のドット積を計算すると `m - n`（相対距離）のみに依存した値になる
- **Llama、Qwen、DeepSeek、Mistral** など主要なオープンソース LLM のほぼすべてで採用されているが、**学習時の長さ $L$ を大きく超えた長さでは精度が著しく劣化する**という問題が実験的に確認されており、これが次に述べるスケーリング手法の動機となっている

### RoPE スケーリング手法

**Position Interpolation（Chen et al., Meta, 2023）** は最もシンプルなアプローチで、目標長 `L_target` のすべての位置インデックスを学習時の最大長 `L_train` の範囲に収まるようにリスケールする（`pos' = pos × (L_train / L_target)`）

- 外挿（未知の範囲への適用）よりも**内挿（既知の範囲内への補間）**の方がモデルにとって扱いやすいため有効だが、高周波成分（局所的な位置情報を担う次元）が圧縮されて精度が劣化し、ファインチューニングも必要になる

**NTK-aware Interpolation（bloc97, 2023）** は全周波数を一律にスケールするのではなく、**次元ごとに異なるスケーリング係数を適用**する
低周波次元（長距離の位置を担当）は大きくスケールし、高周波次元（局所的な位置を担当）はスケールを小さく抑えることで、高周波成分の情報損失を回避しつつ長距離の位置情報を拡張できる

- ファインチューニング不要でも一定の性能が得られる利点がある

**YaRN（Yet another RoPE extensioN, Peng et al., ICLR 2024）** は現時点でコンテキスト拡張の事実上の標準手法である
NTK-by-Parts（区分線形スケーリング）に加えて**アテンション温度スケーリング**を組み合わせた

- コンテキスト長が伸びると Q・K のドット積の分布が変化し softmax の出力がフラットになって情報が拡散してしまう問題を、温度パラメータで補正する

- 主な成果として、**同等のコンテキスト品質を達成するためのファインチューニングトークン数が従来手法の 1/10**、**学習ステップ数が 2.5 倍削減**を実現した
  - **Qwen、DeepSeek、Llama** ベースのモデルのほぼすべてが YaRN ベースのコンテキスト拡張を採用している
  - **LongRoPE（Microsoft, 2024）** はさらに一歩進んで、8K → 64K → 512K → 2M という段階的拡張戦略で **2M トークン**への到達を可能にした

### LoRA ベースのコンテキスト拡張

上記の RoPE スケーリング手法とファインチューニングを組み合わせる際、**LoRA** を使うことが実用上の標準になっている

フルパラメータのファインチューニングは大規模モデルでは現実的でない場合が多いため、以下のパイプラインが典型的である：

1. ベースモデルの RoPE パラメータを YaRN スケーリングで変更
2. 長文書コーパス（書籍・論文・長い会話など）で LoRA アダプタを学習
3. 推論時はベースモデル + LoRA アダプタを組み合わせて使用

フルファインチューニングの数十分の一のコストでコンテキスト拡張が実現できる

### その他のアプローチ

**ALiBi（Attention with Linear Biases, Press et al., 2022）** は位置埋め込みを廃止し、アテンションスコアに**距離に比例した線形ペナルティ**を直接加算する方式で、学習時の長さを超えた位置に対しても自然に外挿できる

**TTT（Test-Time Training, 2025）** は発想の異なるアプローチで、長いコンテキストを KV キャッシュに保持するのではなく、**推論時にそのコンテキストの next-token prediction タスクで重みを更新しコンテキストの情報をモデルのパラメータに「記憶」させる**
- KV キャッシュのサイズがコンテキスト長 $n$ に対して $O(n)$ で増加する通常の推論に対し、TTT では $O(1)$ の固定サイズで推論でき、**128K トークンで 2.7 倍の高速化**、**2M トークンで 35 倍の高速化**が報告されている
- ただし推論時に勾配計算・重み更新が必要になるため、2025 年時点ではまだ研究段階である

### 現状（2025–2026）

コンテキスト長の拡大は過去数年で劇的に進んだ：
- GPT-2（2019, 1,024 トークン）→ GPT-4（2023, 128K）→ Gemini 1.5 Pro（2024, 1M）→ Grok・Gemini 2.0（2024–2025, 1〜2M）

現在の主流は **YaRN スケーリング + LoRA ファインチューニング**で、多くのオープンソースモデルが 128K〜1M トークンへの対応を実現している

さらに **Ring Attention** を組み合わせることで分散システム上での 10M トークン以上の対応も技術的に可能になっている

### ハイブリッドアーキテクチャ ― Attention と代替機構の融合

#### なぜハイブリッドか

純粋な Transformer は優れたアーキテクチャだが、大規模・長コンテキスト��推論ではアテンション計算のコストが $O(n^2)$ で増加し、KV キャッシュのサイズが $O(n \cdot d \cdot \text{layers})$ で増加するという本質的な課題を抱えている

一方、純粋な SSM/RNN（Mamba、RWKV など）は推論コスト $O(1)$ per step で KV キャッシュが不要という圧倒的な効率性を持つが、「コンテキスト��の特定情報を正確に検索・参照する」タスク（いわゆる needle-in-a-haystack 問題）での精度がアテンションより劣る場合がある
固定サイズの隠れ状態にすべての過去情報を圧縮するため、情報の損失が避けられないためである

ハイブリッドアーキテクチャの発想は、**アテンションが得意な部分（正確な情報検索）にはアテンション層を使い、それ以外の大半の計算は効率的な SSM/RNN 層で処理する**というものである

#### 主要なハイブリッドモデル

**Jamba（AI21 Labs, 2024）** は初の大規模本番投入ハイブリッドモデルとして注目を集めた
**Transformer アテンション層と Mamba SSM 層を約 1:7 の比率でインタリーブ**（8 層ごとに 1 層のアテンション）し、一部の層に MoE を追加してパラメータ効率をさらに高めている

Jamba-1.5-Large（94B アクティブパラメータ）と Jamba-1.5-Mini（12B アクティブパラメータ）の二構成があり、**スループット Mixtral-8x7B 比 3 倍向上**、**実効コンテキスト長 256K トークン**を達成しつつ品質は同クラスの純粋 Transformer モデルと同等を維持する

なぜ 1:7 という比率が有効なのか：実験的に全層の 1/8 をアテンション層にするだけで needle-in-a-haystack タスクの精度がほぼ純粋 Transformer と同等になることがわかっている

残りの 7/8 の層を Mamba に置き換えることで推論コストを大幅に削減できる

**StripedHyena（Together AI, 2023）** は Transformer アテンション層とゲート付き畳み込み/SSM 層を**縞模様（striped）のパターン**で交互に配置するアーキテクチャで、長コンテキスト処理の効率化を主な目的として設計されている

**Zamba（Zyphra, 2024）** は Mamba SSM をバックボーンとし、一定間隔で**共有アテンション層（shared attention layers）**を挿入する
- 「共有」とは複数の位置で同一の重みを持つアテンション層を使い回すことで、パラメータ数を増やさずにアテンションの恩恵を得られる
- 小〜中規模モデル（1B〜7B クラス）でのエッジ推論を意識した設計が特徴である

**RecurrentGemma（Google DeepMind, 2024）** は **Griffin アーキテクチャ**を採用し、**ゲート付き線形再帰（Gated Linear Recurrence）**と**局所アテンション（Local Attention）**を組み合わせる

グローバルな文脈は再帰で、局所的な精密処理はアテンションで担当する設計になっている

#### ハイブリッド設計の原則

研究の蓄積から、効果的なハイブリッド設計のための経験則が見えてきた：

- **アテンション層**は needle-in-a-haystack 型のタスクで本領を発揮する。$O(n^2)$ のコストで全トークン間の関係を直接計算できるため、正確な検索・参照が可能になる
- **SSM/RNN 層**は連続的な依存関係の処理が得意で、文脈の流れを追いながら情報を統合する処理の大部分を $O(1)$ per step で効率的に処理できる
- **比率の経験則**として、**1 アテンション層 : 4〜8 SSM 層**でほとんどのタスクに十分な品質が得られる
- **MoE との組み合わせ**は直交的な手法として重ねることができ、ハイブリッド × MoE の組み合わせ（Jamba がその好例）は現在最も活発に研究されているアーキテクチャ空間のひとつである

#### 展望（2025–2026 以降）

主要な AI 研究機関がハイブリッドをリリース済みまたは開発中であり、**純粋 Transformer の独占的地位は徐々に侵食されつつある**

SSM の品質が急速に向上し、多くのベンチマークが Transformer 同等に達しつつあり、エッジ推論といったモバイル展開での需要が高まる中で $O(1)$ 推論コストのハイブリッドモデルの採用が加速する見通しである

最も根本的な研究課題は「**最適なアーキテクチャの混合比を自動的に学習できるか**」で、**NAS（Neural Architecture Search）のハイブリッド LLM 版**として最適な層の組み合わせを自動探索する手法の研究が始まっている

### 実装演習 ― 各効率化手法を GuppyLM に適用する

以下では、上で解説した効率化手法のエッセンスを GuppyLM のスケール（`d_model=384`, `n_heads=6`）で実装し、動作を確認する
最後に複数の手法を統合した **Guppy-LM+** を構築して学習・推論を行う

In [15]:
# ── Attention 効率化: Sliding Window Attention & Linear Attention ──

class SlidingWindowAttention(nn.Module):
    """Longformer 風の局所窓アテンション
    各トークンは直前 window_size トークンのみに注意する  O(n * w)
    """
    def __init__(self, d_model, n_heads, window_size=16, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.window_size = window_size
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), 1)
        window = torch.tril(torch.ones(T, T, device=x.device, dtype=torch.bool),
                            -self.window_size - 1)
        att = att.masked_fill(causal | window, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))


class LinearAttention(nn.Module):
    """カーネル近似 O(n) Attention: phi(x) = elu(x)+1"""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads, self.head_dim = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.resid_drop = nn.Dropout(dropout)

    @staticmethod
    def feature_map(x):
        return F.elu(x) + 1

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        q, k = self.feature_map(q), self.feature_map(k)
        kv = torch.einsum('bhnd,bhne->bhnde', k, v).cumsum(dim=2)
        z  = k.cumsum(dim=2)
        y  = torch.einsum('bhnd,bhnde->bhne', q, kv)
        y  = y / torch.einsum('bhnd,bhnd->bhn', q, z).unsqueeze(-1).clamp(min=1e-6)
        y  = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))


# デモ: 計算時間比較
x_demo = torch.randn(1, MAX_SEQ_LEN - 1, 384, device=device)
attn_std = MultiHeadSelfAttention(384, 6).to(device)
attn_sw  = SlidingWindowAttention(384, 6, window_size=16).to(device)
attn_lin = LinearAttention(384, 6).to(device)
for name, mod in [('Standard', attn_std), ('SlidingWindow-16', attn_sw), ('Linear', attn_lin)]:
    t0 = time.time()
    for _ in range(10): _ = mod(x_demo)
    print(f'{name:20s}: {(time.time()-t0)/10*1000:.1f} ms / forward')

Standard            : 0.8 ms / forward
SlidingWindow-16    : 1.0 ms / forward
Linear              : 9.7 ms / forward


In [16]:
# ── KV キャッシュ削減: Grouped-Query Attention (GQA) ──

class GroupedQueryAttention(nn.Module):
    """GQA: n_kv_heads=1 -> MQA, n_kv_heads=n_heads -> MHA"""
    def __init__(self, d_model, n_heads, n_kv_heads=2, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.wq = nn.Linear(d_model, n_heads * self.head_dim)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), 1)
        att = att.masked_fill(mask, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_drop(self.proj(y))

d, h, kv_h = 384, 6, 2
print(f'MHA   KV cache / token: {2*h*(d//h):4d} floats')
print(f'GQA-{kv_h} KV cache / token: {2*kv_h*(d//h):4d} floats  (削減率 {1-kv_h/h:.0%})')

MHA   KV cache / token:  768 floats
GQA-2 KV cache / token:  256 floats  (削減率 67%)


In [17]:
# ── モデル圧縮: 量子化 & Wanda 風プルーニング ──
import copy

def quantize_tensor(w, bits=8):
    qmax = 2 ** (bits - 1) - 1
    scale = w.abs().max() / qmax
    w_q = (w / scale).round().clamp(-qmax, qmax)
    return w_q * scale, scale

with torch.no_grad():
    w = model.blocks[0].ffn.fc1.weight.clone()
    w_q8, _ = quantize_tensor(w, 8)
    w_q4, _ = quantize_tensor(w, 4)
    err8 = (w - w_q8).abs().mean().item()
    err4 = (w - w_q4).abs().mean().item()
    print('=== 量子化デモ (blocks[0].ffn.fc1) ===')
    print(f'INT8 mean abs error: {err8:.6f}')
    print(f'INT4 mean abs error: {err4:.6f}  ({err4/err8:.1f}x worse)')

def wanda_prune(m, sparsity=0.5):
    n_pruned = n_total = 0
    with torch.no_grad():
        for name, mod in m.named_modules():
            if not isinstance(mod, nn.Linear) or 'head' in name:
                continue
            W = mod.weight.data
            score = W.abs() * W.abs().mean(dim=0, keepdim=True)
            thr = torch.quantile(score.flatten(), sparsity)
            mask = score > thr
            mod.weight.data *= mask.float()
            n_pruned += (~mask).sum().item()
            n_total += mask.numel()
    return n_pruned, n_total

print('\n=== Wanda 風プルーニング (50%) ===')
loss_before = estimate_loss(20)
model_pruned = copy.deepcopy(model)
p, t = wanda_prune(model_pruned, 0.5)
print(f'Pruned {p:,}/{t:,} weights ({p/t:.1%})')
_orig = model; model = model_pruned
loss_after = estimate_loss(20)
model = _orig
print(f'Before: loss={loss_before:.4f}  After: loss={loss_after:.4f}  (delta={loss_after-loss_before:+.4f})')

=== 量子化デモ (blocks[0].ffn.fc1) ===
INT8 mean abs error: 0.000231
INT4 mean abs error: 0.004179  (18.1x worse)

=== Wanda 風プルーニング (50%) ===
Pruned 3,538,944/7,077,888 weights (50.0%)
Before: loss=0.3750  After: loss=0.3764  (delta=+0.0014)


In [18]:
# ── MoE: Mixture-of-Experts FFN ──

class MoEFeedForward(nn.Module):
    """Top-k routing MoE: n_experts 個の FFN から top_k を選択"""
    def __init__(self, d_model, ffn_hidden, n_experts=4, top_k=2, dropout=0.1):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        self.router = nn.Linear(d_model, n_experts, bias=False)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, ffn_hidden), nn.ReLU(),
                          nn.Linear(ffn_hidden, d_model), nn.Dropout(dropout))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        B, T, C = x.shape
        x_flat = x.view(-1, C)
        logits = self.router(x_flat)
        topk_w, topk_idx = logits.topk(self.top_k, dim=-1)
        gates = F.softmax(topk_w, dim=-1)
        out = torch.zeros_like(x_flat)
        for k in range(self.top_k):
            for e in range(self.n_experts):
                mask = (topk_idx[:, k] == e)
                if mask.any():
                    out[mask] += gates[mask, k:k+1] * self.experts[e](x_flat[mask])
        return out.view(B, T, C)

moe_ffn   = MoEFeedForward(384, 192, n_experts=4, top_k=2)
dense_ffn = FeedForward(384, 768)
print(f'Dense FFN (768-hidden):  {sum(p.numel() for p in dense_ffn.parameters()):>8,} params')
print(f'MoE-4x192  (top-2):     {sum(p.numel() for p in moe_ffn.parameters()):>8,} params')

Dense FFN (768-hidden):   590,976 params
MoE-4x192  (top-2):      593,664 params


In [19]:
# ── LoRA: Low-Rank Adaptation ──

class LoRALinear(nn.Module):
    """既存 Linear に低ランク更新 delta_W = BA を追加"""
    def __init__(self, original: nn.Linear, rank=4, alpha=1.0):
        super().__init__()
        self.original = original
        original.weight.requires_grad_(False)
        if original.bias is not None:
            original.bias.requires_grad_(False)
        d_out, d_in = original.weight.shape
        self.scaling = alpha / rank
        self.lora_A = nn.Parameter(torch.randn(rank, d_in) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(d_out, rank))

    def forward(self, x):
        return self.original(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling

def apply_lora(m, rank=4, alpha=1.0, targets=('qkv', 'proj')):
    count = 0
    for _, mod in m.named_modules():
        for t in targets:
            if hasattr(mod, t) and isinstance(getattr(mod, t), nn.Linear):
                setattr(mod, t, LoRALinear(getattr(mod, t), rank, alpha))
                count += 1
    return count

model_lora = copy.deepcopy(model)
n_adapted = apply_lora(model_lora, rank=4)
total     = sum(p.numel() for p in model_lora.parameters())
trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f'LoRA rank=4 applied to {n_adapted} layers')
print(f'Total:     {total:>10,}')
print(f'Trainable: {trainable:>10,}  ({trainable/total:.2%})')
print(f'Frozen:    {total-trainable:>10,}  ({(total-trainable)/total:.2%})')

LoRA rank=4 applied to 12 layers
Total:      8,136,960
Trainable:  4,588,800  (56.39%)
Frozen:     3,548,160  (43.61%)


In [21]:
# ── コンテキスト長拡張: RoPE & Position Interpolation ──

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=128, base=10000.0):
        super().__init__()
        freqs = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer('freqs', freqs)
        self.max_seq_len = max_seq_len

    def forward(self, seq_len, device, scale=1.0):
        t = torch.arange(seq_len, device=device, dtype=self.freqs.dtype) / scale
        emb = torch.outer(t, self.freqs)
        return emb.cos(), emb.sin()

def apply_rope(q, k, cos, sin):
    d2 = q.shape[-1] // 2
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    q1, q2 = q[..., :d2], q[..., d2:]
    k1, k2 = k[..., :d2], k[..., d2:]
    return (torch.cat([q1*cos - q2*sin, q2*cos + q1*sin], -1),
            torch.cat([k1*cos - k2*sin, k2*cos + k1*sin], -1))

rope = RotaryEmbedding(64, 128).to(device)
cos, sin = rope(128, device)
q = torch.randn(1, 1, 128, 64, device=device)
k = torch.randn(1, 1, 128, 64, device=device)
q_r, k_r = apply_rope(q, k, cos, sin)
print('RoPE: 位置 10 と各位置の内積')
for j in [10, 20, 30, 50]:
    dot = (q_r[0,0,10] * k_r[0,0,j]).sum().item()
    print(f'  pos(10,{j:2d})  dist={j-10:2d}  dot={dot:+.3f}')
cos2, sin2 = rope(256, device, scale=2.0)
print(f'\nPosition Interpolation: 128 -> 256 (scale=2.0)')
print(f'  cos shape: {cos2.shape}  (256 positions mapped into 128 range)')

RoPE: 位置 10 と各位置の内積
  pos(10,10)  dist= 0  dot=+1.057
  pos(10,20)  dist=10  dot=-3.707
  pos(10,30)  dist=20  dot=+6.933
  pos(10,50)  dist=40  dot=-6.417

Position Interpolation: 128 -> 256 (scale=2.0)
  cos shape: torch.Size([256, 32])  (256 positions mapped into 128 range)


#### 統合: Guppy-LM+（GQA + MoE + RoPE）

上記の手法を組み合わせた拡張モデルを定義する

- **GQA**（`n_kv_heads=2`）で KV キャッシュを 67% 削減
- **MoE FFN**（4 エキスパート × 192-hidden、top-2）で疎な活性化
- **RoPE** で相対位置表現 + Position Interpolation によるコンテキスト拡張

In [22]:
# ── 統合モデル: Guppy-LM+ ──

class GQAWithRoPE(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, dropout=0.1):
        super().__init__()
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.head_dim = d_model // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.wq = nn.Linear(d_model, n_heads * self.head_dim)
        self.wk = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.wv = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.proj = nn.Linear(d_model, d_model)
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.wq(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, cos, sin)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), 1)
        att = att.masked_fill(mask, float('-inf'))
        att = self.attn_drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_drop(self.proj(y))

class GuppyPlusBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, ffn_hidden,
                 n_experts, top_k, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = GQAWithRoPE(d_model, n_heads, n_kv_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = MoEFeedForward(d_model, ffn_hidden, n_experts, top_k, dropout)
    def forward(self, x, cos, sin):
        x = x + self.attn(self.ln1(x), cos, sin)
        x = x + self.ffn(self.ln2(x))
        return x

class GuppyLMPlus(nn.Module):
    """GQA + MoE + RoPE を統合した GuppyLM 拡張版"""
    def __init__(self, vocab_size, d_model=384, n_layers=6, n_heads=6,
                 n_kv_heads=2, ffn_hidden=192, n_experts=4, top_k=2,
                 max_seq_len=128, dropout=0.1, pad_id=0, rope_base=10000.0):
        super().__init__()
        self.max_seq_len, self.pad_id = max_seq_len, pad_id
        self.head_dim = d_model // n_heads
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len, rope_base)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            GuppyPlusBlock(d_model, n_heads, n_kv_heads, ffn_hidden,
                           n_experts, top_k, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        self.apply(self._init_w)
    @staticmethod
    def _init_w(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0, 0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0, 0.02)
    def forward(self, idx, targets=None, rope_scale=1.0):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx))
        cos, sin = self.rope(T, idx.device, scale=rope_scale)
        for blk in self.blocks: x = blk(x, cos, sin)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=self.pad_id)
        return logits, loss
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None,
                 eos_id=None, rope_scale=1.0):
        self.eval()
        for _ in range(max_new_tokens):
            idx_c = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_c, rope_scale=rope_scale)
            logits = logits[:, -1, :] / max(1e-8, temperature)
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float('-inf')
            nxt = torch.multinomial(F.softmax(logits, dim=-1), 1)
            idx = torch.cat([idx, nxt], dim=1)
            if eos_id is not None and nxt.item() == eos_id: break
        return idx

model_plus = GuppyLMPlus(
    vocab_size=VOCAB_SIZE, d_model=384, n_layers=6, n_heads=6,
    n_kv_heads=2, ffn_hidden=192, n_experts=4, top_k=2,
    max_seq_len=MAX_SEQ_LEN, dropout=0.1, pad_id=PAD_ID).to(device)

print(f'GuppyLM  (original):      {count_params(model)/1e6:.2f} M')
print(f'GuppyLM+ (GQA+MoE+RoPE): {count_params(model_plus)/1e6:.2f} M')
print(f'  n_kv_heads=2 -> KV cache 67% 削減')
print(f'  MoE 4x192 top-2')
print(f'  RoPE -> Position Interpolation 可能')

GuppyLM  (original):      8.08 M
GuppyLM+ (GQA+MoE+RoPE): 6.87 M
  n_kv_heads=2 -> KV cache 67% 削減
  MoE 4x192 top-2
  RoPE -> Position Interpolation 可能


In [23]:
# ── Guppy-LM+ 学習 (2000 ステップ) & サンプル対話 ──

PLUS_STEPS = 2000
opt_plus = torch.optim.AdamW(model_plus.parameters(), lr=3e-4,
                              betas=(0.9, 0.95), weight_decay=0.1)
scaler_plus = torch.amp.GradScaler('cuda', enabled=use_amp)
it_plus = endless(train_loader)
model_plus.train()
t0 = time.time(); running = 0.0
for step in range(1, PLUS_STEPS + 1):
    lr_now = lr_at(step)
    for g in opt_plus.param_groups: g['lr'] = lr_now
    x, y = next(it_plus)
    x, y = x.to(device), y.to(device)
    opt_plus.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', enabled=use_amp):
        _, loss = model_plus(x, y)
    scaler_plus.scale(loss).backward()
    scaler_plus.unscale_(opt_plus)
    torch.nn.utils.clip_grad_norm_(model_plus.parameters(), 1.0)
    scaler_plus.step(opt_plus); scaler_plus.update()
    running += loss.item()
    if step % 200 == 0:
        avg = running / 200; running = 0.0
        print(f'step {step:5d} | loss {avg:.4f} | ppl {math.exp(min(20,avg)):.2f} | {time.time()-t0:.0f}s')

def chat_plus(msg):
    prompt = _format_prompt(msg)
    ids = tokenizer.encode(prompt).ids[-model_plus.max_seq_len:]
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model_plus.generate(idx, max_new_tokens=96, temperature=0.8,
                              top_k=40, eos_id=EOS_ID)
    gen = out[0, len(ids):].tolist()
    if EOS_ID in gen: gen = gen[:gen.index(EOS_ID)]
    return tokenizer.decode(gen).strip()

print('\n=== Guppy-LM+ サンプル対話 ===')
for q in ['Hello! Who are you?', 'What do you eat?', 'Tell me about yourself.']:
    print(f'USER : {q}')
    print(f'GUPPY: {chat_plus(q)}')
    print('-' * 40)

step   200 | loss 4.4026 | ppl 81.66 | 33s
step   400 | loss 2.3614 | ppl 10.61 | 67s
step   600 | loss 1.6433 | ppl 5.17 | 100s
step   800 | loss 1.1604 | ppl 3.19 | 132s
step  1000 | loss 0.8436 | ppl 2.32 | 165s
step  1200 | loss 0.6802 | ppl 1.97 | 197s
step  1400 | loss 0.5907 | ppl 1.81 | 231s
step  1600 | loss 0.5379 | ppl 1.71 | 264s
step  1800 | loss 0.5078 | ppl 1.66 | 296s
step  2000 | loss 0.4839 | ppl 1.62 | 329s

=== Guppy-LM+ サンプル対話 ===
USER : Hello! Who are you?
GUPPY: you are my only friend who talks back.
----------------------------------------
USER : What do you eat?
GUPPY: food is the best thing. i like the the soft ones best.
----------------------------------------
USER : Tell me about yourself.
GUPPY: my mouth are small but they work perfectly.
----------------------------------------


### Guppy-LM+ 用 ipywidgets チャット UI

GQA + MoE + RoPE を統合した縮約モデル Guppy-LM+(`model_plus`)に対しても、同じ要領で ipywidgets ベースのチャット UI を用意する

内部では入力欄と送信ボタンを持つが、先のチャットUIとの衝突を避けるため、`chat_completion` の代わりに `chat_plus` を呼び出してモデルの応答を得る

ipywidgetsは カーネル(Python)とブラウザ(JS)を comm チャネルで同期 する仕組み
- ウィジェットの状態(value, description等)は両側にミラーされ、WebSocket経由でJSONメッセージとして同期
- ブラウザで Enter/Click が起きると → カーネルに通知 → on_click に登録したPython関数が発火 → 結果(log出力など)が再びブラウザへ反映
- 「UIはブラウザ、モデル推論はColabのGPU」という分業が自然に成り立つ  

In [24]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    log_plus = widgets.Output(layout={'border': '1px solid #888', 'height': '260px', 'overflow': 'auto'})
    inp_plus = widgets.Text(placeholder='Say something to Guppy-LM+ ...', layout={'width': '70%'})
    btn_plus = widgets.Button(description='Send', button_style='success')

    def on_send_plus(_=None):
        msg = inp_plus.value.strip()
        if not msg:
            return
        inp_plus.value = ''
        with log_plus:
            print(f'USER  : {msg}')
            reply = chat_plus(msg)
            print(f'GUPPY+: {reply}')
            print('-' * 40)

    btn_plus.on_click(on_send_plus)
    inp_plus.on_submit(on_send_plus)
    display(widgets.VBox([log_plus, widgets.HBox([inp_plus, btn_plus])]))
except Exception as e:
    print('ipywidgets UI is not available in this environment:', e)

## まとめ

本ノートブックでは、極小 LLM **GuppyLM** を PyTorch でゼロから再実装し、6 万件の QA データを ChatML に整形して SFT するところまでを扱った

さらに、同じ Transformer アーキテクチャが数百 B クラスにスケールする際に直面する効率化の課題と、それを克服する最先端の手法群を体系的に整理した

ポイントを振り返ると次の通りである

**GuppyLM の実装**
- GuppyLM は **8.7M パラメータ** の単一構成しか存在しない、教育・実験向けの極小モデル
- アーキテクチャは pre-norm Transformer + ReLU FFN + 学習可能な位置埋め込み + weight tying という、GPT 系の最小構成
- トークナイザ・データセットとも公式が Hugging Face と GitHub 上に公開しており、数行のコードで完全に同一のパイプラインを再現可能
- 推論は ChatML の `<|im_start|>assistant\n` を prefix に与える典型的な手順で、`<|im_end|>` を EOS として扱う

**LLM 効率化手法**
- Attention の $O(n^2)$ ボトルネックに対し、Sparse / Linear / RWKV / SSM（Mamba）と多様なアプローチが存在し、それぞれ異なるトレードオフを持つ
- KV キャッシュ削減（GQA、MLA、H2O、PyramidKV）と HW・システム最適化（FlashAttention、PagedAttention、Speculative Decoding）は推論の実用化に不可欠な基盤技術である
- モデル圧縮（量子化・プルーニング・蒸留）、MoE、LoRA 系 PEFT、コンテキスト長拡張（YaRN）は、それぞれ独立に適用でき組み合わせ効果も大きい
- ハイブリッドアーキテクチャ（Jamba 等）が Attention と SSM の長所を融合する新潮流として台頭しつつある

この 8.7M モデルが現代 LLM の「最小の完全版」であることを念頭に置きつつ、効率化手法の全体像を俯瞰することで、Transformer ベースの言語モデルがどのように実用化・大規模化されているかの理解が深まるだろう

## 参考文献

### GuppyLM

1. arman-bd, *GuppyLM ― a tiny 8.7M-parameter chat language model*, GitHub, 2025. <https://github.com/arman-bd/guppylm> (MIT License)
2. arman-bd, *guppylm-60k-generic* dataset, Hugging Face Hub. <https://huggingface.co/datasets/arman-bd/guppylm-60k-generic>
3. arman-bd, *guppylm/docs/tokenizer.json* (BPE tokenizer, vocab=4096). <https://raw.githubusercontent.com/arman-bd/guppylm/main/docs/tokenizer.json>
4. A. Vaswani et al., "Attention Is All You Need", NeurIPS 2017.

### LLM 効率化手法

**Attention の効率化**

5. M. Zaheer et al., "Big Bird: Transformers for Longer Sequences", NeurIPS 2020.
6. I. Beltagy et al., "Longformer: The Long-Document Transformer", arXiv 2020.
7. K. Choromanski et al., "Rethinking Attention with Performers", ICLR 2021.
8. B. Peng et al., "RWKV: Reinventing RNNs for the Transformer Era", EMNLP 2023 Findings.
9. B. Peng et al., "Eagle and Finch: RWKV with Matrix-Valued States and Dynamic Recurrence", arXiv 2024.
10. A. Gu et al., "Efficiently Modeling Long Sequences with Structured State Spaces (S4)", ICLR 2022.
11. A. Gu and T. Dao, "Mamba: Linear-Time Sequence Modeling with Selective State Spaces", arXiv 2023.
12. T. Dao and A. Gu, "Transformers are SSMs: Generalized Models and Efficient Algorithms Through Structured State Space Duality (Mamba-2)", ICML 2024.

**KV キャッシュ削減**

13. N. Shazeer, "Fast Transformer Decoding: One Write-Head is All You Need (MQA)", arXiv 2019.
14. J. Ainslie et al., "GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints", EMNLP 2023.
15. DeepSeek-AI, "DeepSeek-V2: A Strong, Economical, and Efficient Mixture-of-Experts Language Model", arXiv 2024.
16. Z. Zhang et al., "H2O: Heavy-Hitter Oracle for Efficient Generative Inference of Large Language Models", NeurIPS 2023.
17. Y. Li et al., "SnapKV: LLM Knows What You Are Looking For Before Generation", arXiv 2024.
18. Z. Cai et al., "PyramidKV: Dynamic KV Cache Compression based on Pyramidal Information Funneling", arXiv 2024.

**HW・システム最適化**

19. T. Dao et al., "FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness", NeurIPS 2022.
20. T. Dao, "FlashAttention-2: Faster Attention with Better Parallelism and Work Partitioning", ICLR 2024.
21. W. Kwon et al., "Efficient Memory Management for Large Language Model Serving with PagedAttention (vLLM)", SOSP 2023.
22. Y. Leviathan et al., "Fast Inference from Transformers via Speculative Decoding", ICML 2023.
23. L. Zheng et al., "SGLang: Efficient Execution of Structured Language Model Programs", arXiv 2024.
24. H. Liu et al., "Ring Attention with Blockwise Transformers for Near-Infinite Context", ICLR 2024.

**モデル圧縮**

25. E. Frantar et al., "GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers", ICLR 2023.
26. J. Lin et al., "AWQ: Activation-aware Weight Quantization for LLM Compression and Acceleration", MLSys 2024.
27. Google Research, "TurboQuant: Online KV Cache Quantization for Efficient LLM Inference", 2026.
28. E. Frantar and D. Alistarh, "SparseGPT: Massive Language Models Can Be Accurately Pruned in One-Shot", ICML 2023.
29. M. Sun et al., "A Simple and Effective Pruning Approach for Large Language Models (Wanda)", ICLR 2024.
30. S. Muralidharan et al., "Compact Language Models via Pruning and Knowledge Distillation (Minitron)", arXiv 2024.

**MoE**

31. D. Lepikhin et al., "GShard: Scaling Giant Models with Conditional Computation and Automatic Sharding", ICLR 2021.
32. W. Fedus et al., "Switch Transformers: Scaling to Trillion Parameter Models with Simple and Efficient Sparsity", JMLR 2022.
33. Mistral AI, "Mixtral of Experts", arXiv 2024.
34. DeepSeek-AI, "DeepSeek-V3 Technical Report", arXiv 2025.

**LoRA 系 PEFT**

35. E. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models", ICLR 2022.
36. T. Dettmers et al., "QLoRA: Efficient Finetuning of Quantized LLMs", NeurIPS 2023.
37. S.-Y. Liu et al., "DoRA: Weight-Decomposed Low-Rank Adaptation", ICML 2024.
38. J. Zhao et al., "GaLore: Memory-Efficient LLM Training by Gradient Low-Rank Projection", ICML 2024.

**コンテキスト長拡張**

39. J. Su et al., "RoFormer: Enhanced Transformer with Rotary Position Embedding", Neurocomputing 2024.
40. S. Chen et al., "Extending Context Window of Large Language Models via Positional Interpolation", arXiv 2023.
41. B. Peng et al., "YaRN: Efficient Context Window Extension of Large Language Models", ICLR 2024.

**ハイブリッドアーキテクチャ**

42. AI21 Labs, "Jamba: A Hybrid Transformer-Mamba Language Model", arXiv 2024.
43. Google DeepMind, "RecurrentGemma: Moving Past Transformers for Efficient Open Language Models", arXiv 2024.

**その他**

44. Meta AI, *Llama 4 technical report*, 2025.
45. Alibaba, *Qwen 3 technical report*, 2025.
46. Google DeepMind, *Gemma 3 technical report*, 2025.
47. N. Shazeer, "GLU Variants Improve Transformer", arXiv 2020.
48. OpenAI, *ChatML format specification*, 2023.